# 2026 SamplePoint → photo/site species and functional-group cover

This notebook converts the final 2026 SamplePoint classifications into four primary datasets:

1. **photo-level species/surface cover**
2. **site-level species/surface cover**
3. **photo-level functional-group cover**
4. **site-level functional-group cover**

Cover is written as **percent cover (0–100)** to remain compatible with `NCA_Master_Aligned_spp_fg.csv`.

### Identity logic

`2026 Master Survey.xlsx` is authoritative for site/photo identity. The workflow:

- treats SamplePoint sheet 1 as **AW120** and sheet 2 as **Cam5**;
- reconstructs `SourceDatabase` from `Import_Log`;
- matches ordinary records by **source database + camera + photo number**;
- permits sequence correction only when source/survey database counts are equal and all existing direct matches verify the sequence;
- retains the known corrupted-survey photos `AW120 9295–9299` as site `low_43_33`;
- retains `AW120 9915`, attaching it to the unique site containing `9911–9914`;
- excludes stale duplicate source records only when the corrected survey proves that the same camera/photo belongs to another source record;
- writes a full identity-resolution audit before cover aggregation.

This avoids double-counting stale duplicate identities while preserving every valid physical photoplot.

### Site aggregation

A site with `n` usable photos contributes all of its SamplePoint hits:

\[
\mathrm{cover}_k = 100
\frac{\text{hits assigned to class } k}
{\text{all usable hits at the site}}
\]

Thus a 5-photo site normally has 500 hits, a 4-photo site 400 hits, etc.

In [55]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path(r"D:\My Drive\BOP_OCTC_2025")
XLS_DIR = PROJECT_ROOT / "Photos" / "2026" / "Cropped" / "XLS_OutputFiles"

POINT_RAW = XLS_DIR / "2026_SamplePoint_Aggregated" / "26PointRaw_Resolved.xlsx"


MASTER_SURVEY = PROJECT_ROOT / "2026 Master Survey.xlsx"
SURVEY_SHEET = "Sheet1"

NCA_MASTER_CANDIDATES = [
    Path(
        r"N:\Data02\projects-active\BOPclassification_2025"
        r"\Field Data (Processed)\2026"
        r"\NCA_Master_Aligned_spp_fg.csv"
    ),
    Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv"),
]
NCA_MASTER = next(
    (p for p in NCA_MASTER_CANDIDATES if p.exists()),
    NCA_MASTER_CANDIDATES[0],
)

OUTPUT_DIR = XLS_DIR / "2026_SamplePoint_Aggregated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2026

PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]

for path in [POINT_RAW, MASTER_SURVEY]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Point classifications:", POINT_RAW)
print("Master survey:        ", MASTER_SURVEY)
print("Output directory:     ", OUTPUT_DIR)
print("Legacy NCA master:    ", NCA_MASTER, "(exists)" if NCA_MASTER.exists() else "(not found)")

Point classifications: D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\26PointRaw_Resolved.xlsx
Master survey:         D:\My Drive\BOP_OCTC_2025\2026 Master Survey.xlsx
Output directory:      D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated
Legacy NCA master:     C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv (exists)


## 1. Helpers and canonical labels

The species/surface outputs retain individual labels, but obvious symbol/format variants are canonicalized before aggregation so 2026 can align with the historical NCA table.

In [56]:
def clean_text(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return None if text == "" or text.lower() == "nan" else text


def normalize_plot_id(value):
    text = clean_text(value)
    if text is None:
        return None
    if re.fullmatch(r"-?\d+\.0", text):
        text = text[:-2]
    return text


def resolve_plot_id(row):
    for col in ["Plot ID", "Plot ID Incidental"]:
        if col in row.index:
            value = normalize_plot_id(row[col])
            if value is not None:
                return value
    return None


def normalize_camera(value):
    text = clean_text(value)
    if text is None:
        return None
    low = text.lower()
    if "aw120" in low:
        return "AW120"
    if "cam5" in low or "cam 5" in low or "camera5" in low or "camera 5" in low:
        return "Cam5"
    return None


def normalize_database(value):
    text = clean_text(value)
    if text is None:
        return None
    return re.sub(r"\.xlsx?$", "", text.lower())


def extract_photo_number(value):
    if value is None:
        return None
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return None
        if float(value).is_integer():
            return int(value)

    text = str(value).strip()
    match = re.search(r"DSCN0*(\d+)", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    match = re.fullmatch(r"0*(\d+)(?:\.0+)?", text)
    if match:
        return int(match.group(1))

    return None


LABEL_ALIASES = {
    "BAREGRND": "BAREGROUND",
    "BIOCRST": "BIOCRUST",
    "UNKPLANT": "UNK.PLANT",
    "BAPR": "BAPR5",
    "HECO26": "HECO",
    # Field ARSP was used for budsage; current resolved code in this workflow is PIDE4.
    "ARSP": "PIDE4",
}


def canonicalize_label(value):
    text = clean_text(value)
    if text is None:
        return None
    text = text.upper()
    return LABEL_ALIASES.get(text, text)

## 2. Read the final SamplePoint workbook and reconstruct source-database provenance

`Import_Log` contains the row count contributed by each source database. The master sheets were built by appending those database blocks in order, allowing us to recover `SourceDatabase` for each final classification row.

In [58]:
xls = pd.ExcelFile(POINT_RAW)

if len(xls.sheet_names) < 3:
    raise ValueError("Expected AW120, Cam5, and Import_Log sheets.")

AW120_SHEET = xls.sheet_names[0]
CAM5_SHEET = xls.sheet_names[1]

print("Sheet 1 -> AW120:", AW120_SHEET)
print("Sheet 2 -> Cam5: ", CAM5_SHEET)


# ---------------------------------------------------------------------
# Read historical Import_Log
# ---------------------------------------------------------------------

import_log = pd.read_excel(
    POINT_RAW,
    sheet_name="Import_Log"
).copy()

required_log = {
    "Database",
    "RowsToAW120",
    "RowsToCam5",
}

missing_log = required_log - set(import_log.columns)

if missing_log:
    raise ValueError(
        f"Import_Log missing: {sorted(missing_log)}"
    )

import_log["DatabaseNorm"] = (
    import_log["Database"]
    .map(normalize_database)
)


# ---------------------------------------------------------------------
# Make a FINAL in-memory version of the log.
#
# The workbook itself was manually deduplicated after Import_Log was
# created, so the historical row counts are no longer the current
# sheet counts.
#
# Removed from resolved PointRaw:
#   AW120:
#       garrett_database_5  -> 9174            = 1 row
#       garrett_database_8  -> 9435-9439       = 5 rows
#
#   Cam5:
#       cam5_database_15    -> 9161-9165       = 5 rows
# ---------------------------------------------------------------------

final_import_log = import_log.copy()

row_adjustments = {
    ("garrett_database_5", "RowsToAW120"): 1,
    ("garrett_database_8", "RowsToAW120"): 5,
    ("cam5_database_15", "RowsToCam5"): 5,
}

for (database, count_col), n_removed in row_adjustments.items():

    mask = final_import_log["DatabaseNorm"].eq(database)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one Import_Log row for {database}; "
            f"found {mask.sum()}."
        )

    old_n = final_import_log.loc[
        mask,
        count_col
    ].iloc[0]

    old_n = 0 if pd.isna(old_n) else int(old_n)

    final_import_log.loc[
        mask,
        count_col
    ] = old_n - n_removed


# ---------------------------------------------------------------------
# Verify adjusted log totals against the actual resolved workbook
# ---------------------------------------------------------------------

aw_check = pd.read_excel(
    POINT_RAW,
    sheet_name=AW120_SHEET,
)

cam_check = pd.read_excel(
    POINT_RAW,
    sheet_name=CAM5_SHEET,
)

logged_aw = int(
    final_import_log["RowsToAW120"]
    .fillna(0)
    .sum()
)

logged_cam = int(
    final_import_log["RowsToCam5"]
    .fillna(0)
    .sum()
)

print("\nResolved workbook counts:")
print(f"  AW120 sheet: {len(aw_check):,}")
print(f"  Cam5 sheet:  {len(cam_check):,}")

print("\nAdjusted Import_Log counts:")
print(f"  AW120:       {logged_aw:,}")
print(f"  Cam5:        {logged_cam:,}")

if logged_aw != len(aw_check):
    raise ValueError(
        f"Adjusted AW120 Import_Log = {logged_aw:,}, "
        f"but sheet = {len(aw_check):,}."
    )

if logged_cam != len(cam_check):
    raise ValueError(
        f"Adjusted Cam5 Import_Log = {logged_cam:,}, "
        f"but sheet = {len(cam_check):,}."
    )

print("PASS: adjusted source-database counts match resolved workbook.")


# ---------------------------------------------------------------------
# Build samplepoint_photos from RESOLVED PointRaw
# ---------------------------------------------------------------------

point_cols = [
    f"Point{i}"
    for i in range(1, 101)
]

photo_frames = []

for sheet_name, camera, count_col in [

    (
        AW120_SHEET,
        "AW120",
        "RowsToAW120",
    ),

    (
        CAM5_SHEET,
        "Cam5",
        "RowsToCam5",
    ),

]:

    df = pd.read_excel(
        POINT_RAW,
        sheet_name=sheet_name,
    ).copy()

    required = {
        "image",
        "key",
        "Comment",
        "GridSize",
        *point_cols,
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{sheet_name} missing: {sorted(missing)}"
        )


    # Reconstruct source database from the adjusted final counts.
    source_database = []

    for _, rec in final_import_log.iterrows():

        n = (
            0
            if pd.isna(rec[count_col])
            else int(rec[count_col])
        )

        if n > 0:
            source_database.extend(
                [rec["DatabaseNorm"]] * n
            )


    if len(source_database) != len(df):
        raise ValueError(
            f"{sheet_name}: adjusted Import_Log reconstructs "
            f"{len(source_database):,} rows but sheet contains "
            f"{len(df):,}."
        )


    df["Camera"] = camera
    df["MasterSheet"] = sheet_name

    df["MasterExcelRow"] = np.arange(
        2,
        len(df) + 2,
    )

    df["MasterRowID"] = (
        camera
        + ":"
        + df["MasterExcelRow"].astype(str)
    )

    df["SourceDatabase"] = source_database

    df["PhotoNumberRaw"] = (
        df["image"]
        .map(extract_photo_number)
        .astype("Int64")
    )


    if df["PhotoNumberRaw"].isna().any():

        display(
            df.loc[
                df["PhotoNumberRaw"].isna(),
                [
                    "MasterRowID",
                    "image",
                ],
            ]
        )

        raise ValueError(
            "Unreadable SamplePoint photo number."
        )


    df["SourceSequence"] = (
        df.groupby(
            [
                "SourceDatabase",
                "Camera",
            ],
            sort=False,
        )
        .cumcount()
    )

    photo_frames.append(df)


samplepoint_photos = pd.concat(
    photo_frames,
    ignore_index=True,
)


# ---------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------

n_nonblank = (
    samplepoint_photos[
        point_cols
    ]
    .notna()
    .sum(axis=1)
)

if not n_nonblank.eq(100).all():

    display(
        samplepoint_photos.loc[
            ~n_nonblank.eq(100),
            [
                "MasterRowID",
                "Camera",
                "image",
                "SourceDatabase",
            ],
        ].assign(
            n_points=n_nonblank[
                ~n_nonblank.eq(100)
            ]
        )
    )

    raise ValueError(
        "At least one final SamplePoint row "
        "is not 100-point complete."
    )


duplicate_raw_keys = (
    samplepoint_photos[
        samplepoint_photos.duplicated(
            [
                "Camera",
                "PhotoNumberRaw",
            ],
            keep=False,
        )
    ]
    [
        [
            "MasterRowID",
            "SourceDatabase",
            "Camera",
            "PhotoNumberRaw",
            "key",
        ]
    ]
    .sort_values(
        [
            "Camera",
            "PhotoNumberRaw",
        ]
    )
)


print("\nClassification rows by camera:")

display(
    samplepoint_photos[
        "Camera"
    ]
    .value_counts()
    .rename("PhotoRows")
    .to_frame()
)

print(
    f"Total source classification rows: "
    f"{len(samplepoint_photos):,}"
)

print(
    f"Rows participating in camera/photo duplicates: "
    f"{len(duplicate_raw_keys):,}"
)


if len(duplicate_raw_keys):

    display(duplicate_raw_keys)

    raise ValueError(
        "Resolved PointRaw unexpectedly contains duplicate photos."
    )


print(
    "PASS: resolved PointRaw contains 2,269 unique, "
    "100-point-complete classification rows."
)

Sheet 1 -> AW120: AW120_Master
Sheet 2 -> Cam5:  Cam5_Master

Resolved workbook counts:
  AW120 sheet: 1,259
  Cam5 sheet:  1,010

Adjusted Import_Log counts:
  AW120:       1,259
  Cam5:        1,010
PASS: adjusted source-database counts match resolved workbook.

Classification rows by camera:


,PhotoRows
Camera,
AW120,1259
Cam5,1010


Total source classification rows: 2,269
Rows participating in camera/photo duplicates: 0
PASS: resolved PointRaw contains 2,269 unique, 100-point-complete classification rows.


In [59]:
point_long = (
    samplepoint_photos[
        [
            "MasterRowID", "MasterSheet", "MasterExcelRow",
            "SourceDatabase", "SourceSequence", "Camera",
            "PhotoNumberRaw", "image", "key", "Comment", "GridSize",
            *point_cols,
        ]
    ]
    .melt(
        id_vars=[
            "MasterRowID", "MasterSheet", "MasterExcelRow",
            "SourceDatabase", "SourceSequence", "Camera",
            "PhotoNumberRaw", "image", "key", "Comment", "GridSize",
        ],
        value_vars=point_cols,
        var_name="PointField",
        value_name="label_raw",
    )
)

point_long["PointNumber"] = (
    point_long["PointField"].str.extract(r"(\d+)$", expand=False).astype(int)
)
point_long["label"] = point_long["label_raw"].map(canonicalize_label)

if point_long["label"].isna().any():
    display(
        point_long.loc[
            point_long["label"].isna(),
            ["MasterRowID", "PointNumber", "label_raw"],
        ].head(50)
    )
    raise ValueError("Blank label remains after canonicalization.")

expected = len(samplepoint_photos) * 100
if len(point_long) != expected:
    raise ValueError(f"Expected {expected:,} point rows; got {len(point_long):,}.")

print(f"Point rows:        {len(point_long):,}")
print(f"Raw labels:        {point_long['label_raw'].nunique():,}")
print(f"Canonical labels:  {point_long['label'].nunique():,}")

display(
    point_long["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="hits")
)

Point rows:        226,900
Raw labels:        55
Canonical labels:  52


,label,hits
0,BRTE,75939
1,LITTER,33153
2,POSE,24109
3,BIOCRUST,22135
4,BAREGROUND,19911
5,SIAL2,9698
6,ARTR2,9253
7,CETE5,4586
8,KRLA2,3871
9,ROCK,3212


## 3. Build the authoritative survey crosswalk

Each nonblank Center/North/East/South/West cell becomes one survey photo slot. Sites with fewer than five photos are retained naturally.

In [60]:
survey = pd.read_excel(MASTER_SURVEY, sheet_name=SURVEY_SHEET).copy()

required_survey = {
    "Samplepoint Database Name",
    "Camera Number",
    "Plot ID",
    "Plot ID Incidental",
    *PHOTO_COLUMNS,
}
missing_survey = required_survey - set(survey.columns)
if missing_survey:
    raise ValueError(f"Master Survey missing: {sorted(missing_survey)}")

survey["_SurveyRowOrder"] = np.arange(len(survey))
survey["Plot"] = survey.apply(resolve_plot_id, axis=1)
survey["DatabaseNorm"] = survey["Samplepoint Database Name"].map(normalize_database)
survey["Camera"] = survey["Camera Number"].map(normalize_camera)

slot_records = []

for _, site in survey.iterrows():
    plot = site["Plot"]
    if plot is None:
        continue

    for position_order, photo_field in enumerate(PHOTO_COLUMNS):
        photo_number = extract_photo_number(site[photo_field])
        if photo_number is None:
            continue

        slot_records.append({
            "DatabaseNorm": site["DatabaseNorm"],
            "Camera": site["Camera"],
            "PhotoNumber": int(photo_number),
            "Plot": plot,
            "PhotoPosition": photo_field,
            "PhotoPositionOrder": position_order,
            "_SurveyRowOrder": int(site["_SurveyRowOrder"]),
            "Samplepoint Database Name": site["Samplepoint Database Name"],
            "Camera Number": site["Camera Number"],
            "Center Photoplot": site["Center Photoplot"],
            "North Photoplot": site["North Photoplot"],
            "East Photoplot": site["East Photoplot"],
            "South Photoplot": site["South Photoplot"],
            "West Photoplot": site["West Photoplot"],
        })

survey_slots = pd.DataFrame(slot_records)
survey_slots = (
    survey_slots
    .sort_values(["_SurveyRowOrder", "PhotoPositionOrder"])
    .reset_index(drop=True)
)

survey_slots["SurveySequence"] = (
    survey_slots
    .groupby(["DatabaseNorm", "Camera"], sort=False, dropna=False)
    .cumcount()
)

print(f"Survey photo slots:           {len(survey_slots):,}")
print(f"Survey sites with >=1 photo:  {survey_slots['Plot'].nunique():,}")
print(f"Slots with unresolved camera: {survey_slots['Camera'].isna().sum():,}")

camera_photo_plot = (
    survey_slots[survey_slots["Camera"].notna()]
    [["Camera", "PhotoNumber", "Plot"]]
    .drop_duplicates()
)
camera_conflicts = camera_photo_plot[
    camera_photo_plot.duplicated(["Camera", "PhotoNumber"], keep=False)
]
if len(camera_conflicts):
    display(camera_conflicts.sort_values(["Camera", "PhotoNumber", "Plot"]))
    raise ValueError("Corrected survey maps a camera/photo to >1 analytical site.")

db_conflict_mask = (
    survey_slots[survey_slots["Camera"].notna()]
    .duplicated(["DatabaseNorm", "Camera", "PhotoNumber"], keep=False)
)
if db_conflict_mask.any():
    bad = survey_slots[survey_slots["Camera"].notna()].loc[
        db_conflict_mask,
        ["DatabaseNorm", "Camera", "PhotoNumber", "Plot", "PhotoPosition"],
    ]
    display(bad)
    raise ValueError("Duplicate database/camera/photo keys remain in survey.")

print("PASS: survey analytical photo keys are unique.")

Survey photo slots:           2,288
Survey sites with >=1 photo:  458
Slots with unresolved camera: 0
PASS: survey analytical photo keys are unique.


## 4. Resolve SamplePoint rows to survey identities

Resolution order:

1. direct `SourceDatabase + Camera + PhotoNumber`;
2. verified sequence correction for equal-sized database blocks;
3. explicit retained exceptions (`low_43_33` and AW120 `9915`);
4. survey-proven stale duplicate source rows are excluded from ecological aggregation but retained in the audit.

In [61]:
print("POINT_RAW currently points to:")
print(POINT_RAW)

xls_check = pd.ExcelFile(POINT_RAW)

aw_check = pd.read_excel(
    POINT_RAW,
    sheet_name=xls_check.sheet_names[0],
)

cam_check = pd.read_excel(
    POINT_RAW,
    sheet_name=xls_check.sheet_names[1],
)

aw_check["Camera"] = "AW120"
cam_check["Camera"] = "Cam5"

check = pd.concat(
    [aw_check, cam_check],
    ignore_index=True,
)

check["PhotoNumber"] = (
    check["image"]
    .map(extract_photo_number)
)

dups = check[
    check.duplicated(
        ["Camera", "PhotoNumber"],
        keep=False,
    )
]

print(f"\nRows read directly from resolved workbook: {len(check):,}")
print(f"Duplicate camera/photo rows:             {len(dups):,}")

if len(dups):
    display(
        dups[
            ["Camera", "key", "image", "PhotoNumber"]
        ].sort_values(
            ["Camera", "PhotoNumber", "key"]
        )
    )
else:
    print("PASS: resolved workbook itself has no duplicate photo identities.")

POINT_RAW currently points to:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\26PointRaw_Resolved.xlsx

Rows read directly from resolved workbook: 2,269
Duplicate camera/photo rows:             0
PASS: resolved workbook itself has no duplicate photo identities.


In [62]:
# =============================================================================
# 4. RESOLVE CLEAN POINT RAW TO AUTHORITATIVE SURVEY CROSSWALK
#
# PointRaw has already been manually deduplicated/resolved.
# Do NOT perform historical stale-row exclusions or sequence corrections here.
# =============================================================================

resolution = samplepoint_photos[
    [
        "MasterRowID",
        "MasterSheet",
        "MasterExcelRow",
        "SourceDatabase",
        "SourceSequence",
        "Camera",
        "PhotoNumberRaw",
        "image",
        "key",
    ]
].copy()


# =============================================================================
# HARD GATE: PointRaw itself must now be unique
# =============================================================================

raw_duplicates = resolution[
    resolution.duplicated(
        ["Camera", "PhotoNumberRaw"],
        keep=False,
    )
]

if len(raw_duplicates):
    display(
        raw_duplicates.sort_values(
            ["Camera", "PhotoNumberRaw"]
        )
    )
    raise ValueError(
        "Resolved PointRaw still contains duplicate camera/photo identities."
    )

print(
    f"PASS: PointRaw contains "
    f"{len(resolution):,} unique camera/photo records."
)


# =============================================================================
# BUILD SIMPLE CAMERA/PHOTO -> SURVEY CROSSWALK
#
# Section 3 already established that these keys are unique.
# =============================================================================

survey_crosswalk = (
    survey_slots[
        survey_slots["Camera"].notna()
    ]
    [
        [
            "Camera",
            "PhotoNumber",
            "Plot",
            "PhotoPosition",
        ]
    ]
    .drop_duplicates()
    .copy()
)

survey_crosswalk = survey_crosswalk.rename(
    columns={
        "PhotoNumber": "PhotoNumberRaw",
    }
)


# Safety gate in case Section 3 changes later.
survey_dups = survey_crosswalk[
    survey_crosswalk.duplicated(
        ["Camera", "PhotoNumberRaw"],
        keep=False,
    )
]

if len(survey_dups):
    display(survey_dups)
    raise ValueError(
        "Survey crosswalk is not unique by camera/photo."
    )


# =============================================================================
# DIRECT MATCH
# =============================================================================

resolution = resolution.merge(
    survey_crosswalk,
    on=[
        "Camera",
        "PhotoNumberRaw",
    ],
    how="left",
    validate="one_to_one",
)

resolution["Included"] = False
resolution["PhotoNumber"] = pd.NA
resolution["ResolutionType"] = pd.NA
resolution["ExclusionReason"] = pd.NA


direct_mask = resolution["Plot"].notna()

resolution.loc[
    direct_mask,
    "Included",
] = True

resolution.loc[
    direct_mask,
    "PhotoNumber",
] = resolution.loc[
    direct_mask,
    "PhotoNumberRaw",
]

resolution.loc[
    direct_mask,
    "ResolutionType",
] = "survey_camera_photo"


# =============================================================================
# KNOWN EXTRA PHOTO: AW120 9915
#
# This photo was classified even though it was skipped in the survey.
# Infer its site from 9911-9914.
# =============================================================================

mask_9915 = (
    resolution["Plot"].isna()
    & resolution["Camera"].eq("AW120")
    & resolution["PhotoNumberRaw"].eq(9915)
)

if mask_9915.any():

    neighbors = survey_slots[
        survey_slots["Camera"].eq("AW120")
        & survey_slots["PhotoNumber"].isin(
            [9911, 9912, 9913, 9914]
        )
    ].copy()

    candidate_plots = (
        neighbors["Plot"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if len(candidate_plots) != 1:

        display(neighbors)

        raise ValueError(
            "Cannot safely infer AW120 9915 site; "
            f"9911-9914 map to {candidate_plots}."
        )


    plot_9915 = candidate_plots[0]

    used_positions = set(
        neighbors.loc[
            neighbors["Plot"].astype(str).eq(
                plot_9915
            ),
            "PhotoPosition",
        ].dropna()
    )

    missing_positions = [
        p
        for p in PHOTO_COLUMNS
        if p not in used_positions
    ]

    position_9915 = (
        missing_positions[0]
        if len(missing_positions) == 1
        else "Recovered_extra_9915"
    )


    resolution.loc[
        mask_9915,
        "Included",
    ] = True

    resolution.loc[
        mask_9915,
        "Plot",
    ] = plot_9915

    resolution.loc[
        mask_9915,
        "PhotoNumber",
    ] = 9915

    resolution.loc[
        mask_9915,
        "PhotoPosition",
    ] = position_9915

    resolution.loc[
        mask_9915,
        "ResolutionType",
    ] = "manual_skipped_photo_brought_forward"


# =============================================================================
# NOTHING ELSE SHOULD BE UNMATCHED
# =============================================================================

unmatched = resolution[
    ~resolution["Included"]
].copy()

if len(unmatched):

    display(
        unmatched[
            [
                "MasterRowID",
                "SourceDatabase",
                "Camera",
                "PhotoNumberRaw",
                "image",
                "key",
            ]
        ]
    )

    raise ValueError(
        "Resolved PointRaw contains photos that do not match "
        "the authoritative survey crosswalk."
    )


# =============================================================================
# FINAL TYPES / OBJECTS EXPECTED DOWNSTREAM
# =============================================================================

resolution["PhotoNumber"] = pd.to_numeric(
    resolution["PhotoNumber"],
    errors="raise",
).astype("Int64")


included_resolution = resolution.copy()

# Retain this variable so later notebook cells do not break,
# but there should now be zero excluded rows.
excluded_resolution = resolution.iloc[0:0].copy()


# =============================================================================
# FINAL UNIQUE-ID GATE
# =============================================================================

dup_final = included_resolution[
    included_resolution.duplicated(
        ["Camera", "PhotoNumber"],
        keep=False,
    )
]

if len(dup_final):

    display(
        dup_final.sort_values(
            ["Camera", "PhotoNumber"]
        )
    )

    raise ValueError(
        "Resolved analytical photos are not unique by camera/photo."
    )


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 78)
print("IDENTITY RESOLUTION SUMMARY")
print("=" * 78)

print(
    f"Resolved PointRaw photos:   "
    f"{len(resolution):,}"
)

print(
    f"Included analytical photos: "
    f"{len(included_resolution):,}"
)

print(
    f"Excluded source rows:        "
    f"{len(excluded_resolution):,}"
)

print(
    f"Unique sites represented:    "
    f"{included_resolution['Plot'].nunique():,}"
)

print("\nResolution types:")

display(
    resolution[
        "ResolutionType"
    ]
    .value_counts(
        dropna=False
    )
    .rename("Rows")
    .to_frame()
)

print(
    "\nPASS: cleaned PointRaw maps directly "
    "to the authoritative survey."
)

PASS: PointRaw contains 2,269 unique camera/photo records.

IDENTITY RESOLUTION SUMMARY
Resolved PointRaw photos:   2,269
Included analytical photos: 2,269
Excluded source rows:        0
Unique sites represented:    454

Resolution types:


,Rows
ResolutionType,
survey_camera_photo,2268
manual_skipped_photo_brought_forward,1



PASS: cleaned PointRaw maps directly to the authoritative survey.


In [63]:
point_resolved = point_long.merge(
    included_resolution[
        [
            "MasterRowID", "Plot", "PhotoNumber",
            "PhotoPosition", "ResolutionType",
        ]
    ],
    on="MasterRowID",
    how="inner",
    validate="many_to_one",
)

point_resolved["Year"] = YEAR
point_resolved = point_resolved.rename(columns={"image": "image_raw"})
point_resolved["image"] = (
    "DSCN" + point_resolved["PhotoNumber"].astype(str) + "_c.jpg"
)

expected_points = len(included_resolution) * 100
if len(point_resolved) != expected_points:
    raise ValueError(
        f"Expected {expected_points:,} analytical hits; got {len(point_resolved):,}."
    )

hits_per_photo = point_resolved.groupby("MasterRowID").size()
if not hits_per_photo.eq(100).all():
    display(hits_per_photo[~hits_per_photo.eq(100)])
    raise ValueError("At least one analytical photo does not have 100 hits.")

print(f"Analytical point hits: {len(point_resolved):,}")
print(f"Analytical photos:     {point_resolved['MasterRowID'].nunique():,}")
print(f"Sites:                 {point_resolved['Plot'].nunique():,}")

print("\nPhotos per site:")
display(
    point_resolved[["Plot", "MasterRowID"]]
    .drop_duplicates()
    .groupby("Plot")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("n_photos")
    .reset_index(name="n_sites")
)

print("\nlow_43_33 retained photos:")
display(
    included_resolution.loc[
        included_resolution["Plot"].astype(str).eq("low_43_33"),
        ["Camera", "PhotoNumber", "PhotoPosition", "ResolutionType"],
    ]
)

print("\nAW120 9915:")
display(
    included_resolution.loc[
        included_resolution["Camera"].eq("AW120")
        & included_resolution["PhotoNumber"].eq(9915),
        ["Plot", "Camera", "PhotoNumber", "PhotoPosition", "ResolutionType"],
    ]
)

Analytical point hits: 226,900
Analytical photos:     2,269
Sites:                 454

Photos per site:


,n_photos,n_sites
0,4,2
1,5,451
2,6,1



low_43_33 retained photos:


,Camera,PhotoNumber,PhotoPosition,ResolutionType
625,AW120,9295,Center Photoplot,survey_camera_photo
626,AW120,9296,North Photoplot,survey_camera_photo
627,AW120,9297,East Photoplot,survey_camera_photo
628,AW120,9298,South Photoplot,survey_camera_photo
629,AW120,9299,West Photoplot,survey_camera_photo



AW120 9915:


,Plot,Camera,PhotoNumber,PhotoPosition,ResolutionType
394,A3_12,AW120,9915,Center Photoplot,manual_skipped_photo_brought_forward


In [64]:
# ================================================================
# Diagnose sites with >5 analytical photos
# ================================================================

photo_manifest = (
    point_resolved[
        [
            "Plot",
            "MasterRowID",
            "Camera",
            "PhotoNumber",
            "PhotoPosition",
            "SourceDatabase",
        ]
    ]
    .drop_duplicates()
)

photos_per_site = (
    photo_manifest
    .groupby("Plot")
    .size()
    .rename("n_photos")
    .sort_values(ascending=False)
)

problem_sites = photos_per_site[
    photos_per_site > 5
]

print("Sites with >5 photos:")
display(problem_sites.to_frame())


print("\nPhoto records for those sites:")

problem_records = (
    photo_manifest[
        photo_manifest["Plot"].isin(
            problem_sites.index
        )
    ]
    .sort_values(
        [
            "Plot",
            "Camera",
            "PhotoNumber",
        ]
    )
)

display(problem_records)

Sites with >5 photos:


,n_photos
Plot,
A3_12,6



Photo records for those sites:


,Plot,MasterRowID,Camera,PhotoNumber,PhotoPosition,SourceDatabase
389,A3_12,AW120:391,AW120,9910,Center Photoplot,aw120_database_8
390,A3_12,AW120:392,AW120,9911,North Photoplot,aw120_database_8
391,A3_12,AW120:393,AW120,9912,East Photoplot,aw120_database_8
392,A3_12,AW120:394,AW120,9913,South Photoplot,aw120_database_8
393,A3_12,AW120:395,AW120,9914,West Photoplot,aw120_database_8
394,A3_12,AW120:396,AW120,9915,Center Photoplot,aw120_database_8


## 5. Species/surface cover at photo and site scales

The photo table has one row per physical photo, so a site repeats `n` times for its `n` available photos.

The site table pools hits before calculating cover; it does not assume five photos.

In [65]:
def build_photo_cover(points, class_col, class_order=None):
    # One row per analytical photo; cover is percent of that photo's hits.
    photo_meta_cols = [
        "MasterRowID", "Plot", "Year", "Camera", "PhotoNumber",
        "image", "image_raw", "PhotoPosition", "SourceDatabase",
        "ResolutionType", "key", "GridSize",
    ]

    photo_meta = (
        points[photo_meta_cols]
        .drop_duplicates("MasterRowID")
        .copy()
    )

    counts = (
        points.groupby(["MasterRowID", class_col], dropna=False)
        .size()
        .unstack(fill_value=0)
    )
    n_hits = counts.sum(axis=1)
    cover = counts.div(n_hits, axis=0) * 100.0

    if class_order is None:
        class_order = sorted(cover.columns.astype(str))

    for col in class_order:
        if col not in cover.columns:
            cover[col] = 0.0

    cover = cover[class_order]
    cover["n_hits"] = n_hits

    return photo_meta.merge(
        cover.reset_index(),
        on="MasterRowID",
        how="left",
        validate="one_to_one",
    )


def build_site_cover(points, class_col, class_order=None):
    # Pool all hits across all available photos before computing site cover.
    counts = (
        points.groupby(["Plot", class_col], dropna=False)
        .size()
        .unstack(fill_value=0)
    )
    n_hits = counts.sum(axis=1)
    cover = counts.div(n_hits, axis=0) * 100.0

    if class_order is None:
        class_order = sorted(cover.columns.astype(str))

    for col in class_order:
        if col not in cover.columns:
            cover[col] = 0.0

    cover = cover[class_order]

    site_meta = (
        points.groupby("Plot")
        .agg(
            n_photos=("MasterRowID", "nunique"),
            n_hits=("MasterRowID", "size"),
        )
        .reset_index()
    )

    camera_summary = (
        points[["Plot", "Camera"]]
        .drop_duplicates()
        .groupby("Plot")["Camera"]
        .apply(lambda x: x.iloc[0] if len(x) == 1 else "Mixed")
        .rename("Camera")
        .reset_index()
    )

    out = (
        cover.reset_index()
        .merge(site_meta, on="Plot", how="left", validate="one_to_one")
        .merge(camera_summary, on="Plot", how="left", validate="one_to_one")
    )
    out.insert(1, "Year", YEAR)
    return out


species_columns = sorted(point_resolved["label"].unique().tolist())

photo_species = build_photo_cover(
    point_resolved, class_col="label", class_order=species_columns
)
site_species = build_site_cover(
    point_resolved, class_col="label", class_order=species_columns
)

photo_species["total_cover"] = photo_species[species_columns].sum(axis=1)
site_species["total_cover"] = site_species[species_columns].sum(axis=1)

photo_species = photo_species[
    [
        "Plot", "Year", "Camera", "PhotoNumber", "image", "image_raw",
        "PhotoPosition", "SourceDatabase", "ResolutionType",
        "MasterRowID", "key", "GridSize", "n_hits",
        *species_columns, "total_cover",
    ]
]

site_species = site_species[
    [
        "Plot", "Year", "Camera", "n_photos", "n_hits",
        *species_columns, "total_cover",
    ]
]

if not np.allclose(photo_species["total_cover"], 100.0, atol=1e-9):
    raise ValueError("Photo species/surface cover does not sum to 100.")
if not np.allclose(site_species["total_cover"], 100.0, atol=1e-9):
    raise ValueError("Site species/surface cover does not sum to 100.")

print("Photo species/surface:", photo_species.shape)
print("Site species/surface: ", site_species.shape)
display(photo_species.head())
display(site_species.head())

Photo species/surface: (2269, 66)
Site species/surface:  (454, 58)


,Plot,Year,Camera,PhotoNumber,image,image_raw,PhotoPosition,SourceDatabase,ResolutionType,MasterRowID,key,GridSize,n_hits,ACHY,AGCR,AMAC2,ARAR8,ARTR2,ATCA2,ATCO,BAAM4,BAPR5,BAREGROUND,BIOCRUST,BRTE,CADR,CETE5,CHJU,CHVI8,CICY,DEPI,DESO2,DISP,ELEL5,ERCI6,ERNA10,ERTR13,GRSP,HAGL,HECO,KRLA2,LASE,LEPE2,LITTER,MACA2,PHHA,PHLO2,PIDE4,POSE,PSJU3,PSLA3,PSSP6,ROCK,SATR12,SATR12D,SAVE4,SHIT,SIAL2,SIAL2D,TEGL,TRASH,TRDU,UNK.PLANT,VUMI,VUOC,total_cover
0,low_76_0,2026,AW120,9340,DSCN9340_c.jpg,DSCN9340_c.jpg,Center Photoplot,aw120_database_1,survey_camera_photo,AW120:2,1,100.0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
1,low_76_0,2026,AW120,9341,DSCN9341_c.jpg,DSCN9341_c.jpg,North Photoplot,aw120_database_1,survey_camera_photo,AW120:3,2,100.0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
2,low_76_0,2026,AW120,9342,DSCN9342_c.jpg,DSCN9342_c.jpg,East Photoplot,aw120_database_1,survey_camera_photo,AW120:4,3,100.0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,86.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
3,low_76_0,2026,AW120,9343,DSCN9343_c.jpg,DSCN9343_c.jpg,South Photoplot,aw120_database_1,survey_camera_photo,AW120:5,4,100.0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,86.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
4,low_76_0,2026,AW120,9344,DSCN9344_c.jpg,DSCN9344_c.jpg,West Photoplot,aw120_database_1,survey_camera_photo,AW120:6,5,100.0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0


,Plot,Year,Camera,n_photos,n_hits,ACHY,AGCR,AMAC2,ARAR8,ARTR2,ATCA2,ATCO,BAAM4,BAPR5,BAREGROUND,BIOCRUST,BRTE,CADR,CETE5,CHJU,CHVI8,CICY,DEPI,DESO2,DISP,ELEL5,ERCI6,ERNA10,ERTR13,GRSP,HAGL,HECO,KRLA2,LASE,LEPE2,LITTER,MACA2,PHHA,PHLO2,PIDE4,POSE,PSJU3,PSLA3,PSSP6,ROCK,SATR12,SATR12D,SAVE4,SHIT,SIAL2,SIAL2D,TEGL,TRASH,TRDU,UNK.PLANT,VUMI,VUOC,total_cover
0,20260527_mid_17_AGCR,2026,Cam5,5,500,0.0,16.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.4,0.0,61.6,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.2,0.0,0.0,0.0,0.0,5.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
1,20260527_mid_17_erna10,2026,Cam5,5,500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.8,0.0,33.6,0.0,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.4,0.0,0.0,4.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
2,20260601_high_29_erna10,2026,Cam5,5,500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.6,0.0,8.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,100.0
3,20260603_low76_fourwing1,2026,AW120,5,500,0.0,0.0,0.0,0.0,0.0,48.6,0.0,0.0,0.0,2.4,0.0,42.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
4,20260603_low76_fourwing2,2026,AW120,5,500,0.0,0.0,0.0,0.0,0.0,5.2,0.0,0.0,0.0,15.0,0.0,33.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.8,0.0,0.0,0.0,0.0,20.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0


## 6. Functional-group mapping

The coarse groups match the established NCA/MOSAIC structure:

`ARTR_FG`, `BAREGROUND_FG`, `BIOCRUST_FG`, `EAG_FG`, `EF_FG`,
`LITTER_FG`, `NPF_FG`, `OTHER_FG`, `PBG_FG`, `SHRUB_FG`.

Continuity rules include:

- `POSE` → `PBG_FG`
- rock → `BAREGROUND_FG`
- dead `SATR12D` / `SIAL2D` → `LITTER_FG`
- native annual grass/forb and non-bunch perennial grass → `OTHER_FG`
- budsage `PIDE4` (including field `ARSP`) → `ARTR_FG`

The cell displays the complete mapping and stops if any observed canonical label is unmapped.

In [71]:
FG_COLUMNS = [
    "ARTR_FG",
    "BAREGROUND_FG",
    "BIOCRUST_FG",
    "EAG_FG",
    "EF_FG",
    "LITTER_FG",
    "NPF_FG",
    "OTHER_FG",
    "PBG_FG",
    "SHRUB_FG",
]

FG_MAP_2026 = {
    # Sage / sage-like shrubs
    "ARTR2": "ARTR_FG",
    "ARAR8": "ARTR_FG",
    "PIDE4": "ARTR_FG",

    # Surface
    "BAREGROUND": "BAREGROUND_FG",
    "ROCK": "BAREGROUND_FG",
    "BIOCRUST": "BIOCRUST_FG",

    # Exotic annual grass
    "BRTE": "EAG_FG",
    "ERTR13": "EAG_FG",

    # Exotic forb
    "BAPR5": "EF_FG",
    "CADR": "EF_FG",
    "CETE5": "EF_FG",
    "CHJU": "EF_FG",
    "DESO2": "EF_FG",
    "ERCI6": "EF_FG",
    "HAGL": "EF_FG",
    "LASE": "EF_FG",
    "LEPE2": "EF_FG",
    "SATR12": "EF_FG",
    "SIAL2": "EF_FG",
    "TRDU": "EF_FG",

    # Litter / dead vegetation
    "LITTER": "LITTER_FG",
    "SATR12D": "LITTER_FG",
    "SIAL2D": "LITTER_FG",

    # Native perennial forb --> recode to just NF_FG
    "CICY": "NPF_FG",
    "PHHA": "NPF_FG",
    "PHLO2": "NPF_FG",
    "PSLA3": "NPF_FG",

    # Existing coarse scheme has no separate native annual forb/grass
    # or non-bunch perennial-grass group.
    "AMAC2": "NPF_FG", # is a NAF  --> recode to match exotic forb status; no duration flag
    "DEPI": "NPF_FG", #is a NAF --> recode to match exotic forb status; no duration flag
    "DISP": "PBG_FG",
    "MACA2": "NPF_FG",
    "SHIT": "LITTER_FG",
    "TRASH": "LITTER_FG",
    "UNK.PLANT": "OTHER_FG",
    "VUMI": "EAG_FG",
    "VUOC": "EAG_FG",

    # Perennial bunchgrass
    "ACHY": "PBG_FG",
    "AGCR": "PBG_FG",
    "ELEL5": "PBG_FG",
    "HECO": "PBG_FG",
    "POSE": "PBG_FG",
    "PSJU3": "PBG_FG",
    "PSSP6": "PBG_FG",

    # Other shrub
    "ATCA2": "SHRUB_FG",
    "ATCO": "SHRUB_FG",
    "BAAM4": "SHRUB_FG",
    "CHVI8": "SHRUB_FG",
    "ERNA10": "SHRUB_FG",
    "GRSP": "SHRUB_FG",
    "KRLA2": "SHRUB_FG",
    "SAVE4": "SHRUB_FG",
    "TEGL": "SHRUB_FG",
}

observed_labels = set(point_resolved["label"].unique())
unmapped = sorted(observed_labels - set(FG_MAP_2026))
unused = sorted(set(FG_MAP_2026) - observed_labels)

mapping_table = pd.DataFrame(
    sorted(FG_MAP_2026.items(), key=lambda x: (x[1], x[0])),
    columns=["label", "FunctionalGroup"],
)

print("Observed canonical labels:", len(observed_labels))
print("Unmapped observed labels: ", unmapped)
print("Unused map entries:       ", unused)
display(mapping_table)

if unmapped:
    raise ValueError(f"Functional-group mapping incomplete: {unmapped}")

Observed canonical labels: 52
Unmapped observed labels:  []
Unused map entries:        []


,label,FunctionalGroup
0,ARAR8,ARTR_FG
1,ARTR2,ARTR_FG
2,PIDE4,ARTR_FG
3,BAREGROUND,BAREGROUND_FG
4,ROCK,BAREGROUND_FG
5,BIOCRUST,BIOCRUST_FG
6,BRTE,EAG_FG
7,ERTR13,EAG_FG
8,VUMI,EAG_FG
9,VUOC,EAG_FG


In [72]:
# =============================================================================
# REBUILD + QA FUNCTIONAL-GROUP COVER FROM CURRENT FG_MAP_2026
# =============================================================================

# Start fresh from the resolved point-level data.
# Do not reuse an older point_fg object from notebook memory.
point_fg = point_resolved.copy()

point_fg["FunctionalGroup"] = (
    point_fg["label"]
    .map(FG_MAP_2026)
)


# ---------------------------------------------------------------------
# 1. Every observed label must map
# ---------------------------------------------------------------------

unmapped_points = (
    point_fg[
        point_fg["FunctionalGroup"].isna()
    ]
    ["label"]
    .value_counts()
)

if len(unmapped_points):

    display(
        unmapped_points
        .rename("n_hits")
        .to_frame()
    )

    raise ValueError(
        "At least one observed point label failed FG mapping."
    )


# ---------------------------------------------------------------------
# 2. Every mapped value must be one of the ten intended FG columns
# ---------------------------------------------------------------------

observed_fg_values = set(
    point_fg["FunctionalGroup"]
    .dropna()
    .unique()
)

unexpected_fg_values = sorted(
    observed_fg_values
    - set(FG_COLUMNS)
)

print(
    "Observed functional groups:",
    sorted(observed_fg_values),
)

if unexpected_fg_values:

    display(
        point_fg[
            point_fg["FunctionalGroup"].isin(
                unexpected_fg_values
            )
        ]
        [["label", "FunctionalGroup"]]
        .value_counts()
        .rename("n_hits")
        .to_frame()
    )

    raise ValueError(
        "Mapped values outside FG_COLUMNS: "
        f"{unexpected_fg_values}"
    )


# ---------------------------------------------------------------------
# 3. Rebuild photo- and site-level FG cover
# ---------------------------------------------------------------------

photo_fg = build_photo_cover(
    point_fg,
    class_col="FunctionalGroup",
    class_order=FG_COLUMNS,
)

site_fg = build_site_cover(
    point_fg,
    class_col="FunctionalGroup",
    class_order=FG_COLUMNS,
)


# ---------------------------------------------------------------------
# 4. Derived columns
# ---------------------------------------------------------------------

for df in [photo_fg, site_fg]:

    df["SHRUB_total"] = (
        df["ARTR_FG"]
        + df["SHRUB_FG"]
    )

    df["total_cover_fg"] = (
        df[FG_COLUMNS]
        .sum(axis=1)
    )


# ---------------------------------------------------------------------
# 5. Diagnose any remaining sum failures
# ---------------------------------------------------------------------

bad_photo_fg = photo_fg[
    ~np.isclose(
        photo_fg["total_cover_fg"],
        100.0,
        atol=1e-9,
    )
].copy()

bad_site_fg = site_fg[
    ~np.isclose(
        site_fg["total_cover_fg"],
        100.0,
        atol=1e-9,
    )
].copy()


print(
    f"\nPhoto FG total range: "
    f"{photo_fg['total_cover_fg'].min():.12g} "
    f"to "
    f"{photo_fg['total_cover_fg'].max():.12g}"
)

print(
    f"Site FG total range:  "
    f"{site_fg['total_cover_fg'].min():.12g} "
    f"to "
    f"{site_fg['total_cover_fg'].max():.12g}"
)

print(
    f"Photos failing 100%: {len(bad_photo_fg):,}"
)

print(
    f"Sites failing 100%:  {len(bad_site_fg):,}"
)


if len(bad_photo_fg):

    display(
        bad_photo_fg[
            [
                "Plot",
                "Camera",
                "PhotoNumber",
                *FG_COLUMNS,
                "total_cover_fg",
            ]
        ]
        .sort_values("total_cover_fg")
    )

    raise ValueError(
        "Photo FG cover does not sum to 100."
    )


if len(bad_site_fg):

    display(
        bad_site_fg[
            [
                "Plot",
                *FG_COLUMNS,
                "total_cover_fg",
            ]
        ]
        .sort_values("total_cover_fg")
    )

    raise ValueError(
        "Site FG cover does not sum to 100."
    )


# ---------------------------------------------------------------------
# 6. Final column order
# ---------------------------------------------------------------------

photo_fg = photo_fg[
    [
        "Plot",
        "Year",
        "Camera",
        "PhotoNumber",
        "image",
        "image_raw",
        "PhotoPosition",
        "SourceDatabase",
        "ResolutionType",
        "MasterRowID",
        "key",
        "GridSize",
        "n_hits",
        *FG_COLUMNS,
        "SHRUB_total",
        "total_cover_fg",
    ]
]

site_fg = site_fg[
    [
        "Plot",
        "Year",
        "Camera",
        "n_photos",
        "n_hits",
        *FG_COLUMNS,
        "SHRUB_total",
        "total_cover_fg",
    ]
]


print("\nPASS: all functional-group cover sums to 100%.")
print("Photo FG:", photo_fg.shape)
print("Site FG: ", site_fg.shape)

Observed functional groups: ['ARTR_FG', 'BAREGROUND_FG', 'BIOCRUST_FG', 'EAG_FG', 'EF_FG', 'LITTER_FG', 'NPF_FG', 'OTHER_FG', 'PBG_FG', 'SHRUB_FG']

Photo FG total range: 100 to 100
Site FG total range:  100 to 100
Photos failing 100%: 0
Sites failing 100%:  0

PASS: all functional-group cover sums to 100%.
Photo FG: (2269, 25)
Site FG:  (454, 17)


In [73]:
# ================================================================
# Diagnose functional-group cover that does not sum to 100%
# ================================================================

bad_photo_fg = photo_fg.loc[
    ~np.isclose(
        photo_fg["total_cover_fg"],
        100.0,
        atol=1e-9,
    )
].copy()

print(f"Photos failing FG sum: {len(bad_photo_fg):,}")

if len(bad_photo_fg):
    display(
        bad_photo_fg[
            [
                "Plot",
                "Camera",
                "PhotoNumber",
                *FG_COLUMNS,
                "total_cover_fg",
            ]
        ].sort_values("total_cover_fg")
    )

print(
    "\nFG total range:",
    photo_fg["total_cover_fg"].min(),
    "to",
    photo_fg["total_cover_fg"].max(),
)


# If your FG mapping dataframe is called fg_lookup:
print("\nFunctional-group labels in lookup:")
display(
    fg_lookup[
        ["label", "FunctionalGroup"]
    ].sort_values(
        ["FunctionalGroup", "label"]
    )
)

print("\nUnexpected FunctionalGroup values:")
print(
    sorted(
        set(fg_lookup["FunctionalGroup"].dropna())
        - set(FG_COLUMNS)
    )
)

Photos failing FG sum: 0

FG total range: 100.0 to 100.00000000000001

Functional-group labels in lookup:


NameError: name 'fg_lookup' is not defined

In [74]:
point_fg = point_resolved.copy()
point_fg["FunctionalGroup"] = point_fg["label"].map(FG_MAP_2026)

if point_fg["FunctionalGroup"].isna().any():
    raise ValueError("At least one point hit failed FG mapping.")

photo_fg = build_photo_cover(
    point_fg, class_col="FunctionalGroup", class_order=FG_COLUMNS
)
site_fg = build_site_cover(
    point_fg, class_col="FunctionalGroup", class_order=FG_COLUMNS
)

for df in [photo_fg, site_fg]:
    df["SHRUB_total"] = df["ARTR_FG"] + df["SHRUB_FG"]
    df["total_cover_fg"] = df[FG_COLUMNS].sum(axis=1)

photo_fg = photo_fg[
    [
        "Plot", "Year", "Camera", "PhotoNumber", "image", "image_raw",
        "PhotoPosition", "SourceDatabase", "ResolutionType",
        "MasterRowID", "key", "GridSize", "n_hits",
        *FG_COLUMNS, "SHRUB_total", "total_cover_fg",
    ]
]

site_fg = site_fg[
    [
        "Plot", "Year", "Camera", "n_photos", "n_hits",
        *FG_COLUMNS, "SHRUB_total", "total_cover_fg",
    ]
]

if not np.allclose(photo_fg["total_cover_fg"], 100.0, atol=1e-9):
    raise ValueError("Photo FG cover does not sum to 100.")
if not np.allclose(site_fg["total_cover_fg"], 100.0, atol=1e-9):
    raise ValueError("Site FG cover does not sum to 100.")

print("Photo FG:", photo_fg.shape)
print("Site FG: ", site_fg.shape)
display(photo_fg.head())
display(site_fg.head())

Photo FG: (2269, 25)
Site FG:  (454, 17)


,Plot,Year,Camera,PhotoNumber,image,image_raw,PhotoPosition,SourceDatabase,ResolutionType,MasterRowID,key,GridSize,n_hits,ARTR_FG,BAREGROUND_FG,BIOCRUST_FG,EAG_FG,EF_FG,LITTER_FG,NPF_FG,OTHER_FG,PBG_FG,SHRUB_FG,SHRUB_total,total_cover_fg
0,low_76_0,2026,AW120,9340,DSCN9340_c.jpg,DSCN9340_c.jpg,Center Photoplot,aw120_database_1,survey_camera_photo,AW120:2,1,100.0,100,0.0,0.0,0.0,93.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,100.0
1,low_76_0,2026,AW120,9341,DSCN9341_c.jpg,DSCN9341_c.jpg,North Photoplot,aw120_database_1,survey_camera_photo,AW120:3,2,100.0,100,0.0,0.0,0.0,89.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,100.0
2,low_76_0,2026,AW120,9342,DSCN9342_c.jpg,DSCN9342_c.jpg,East Photoplot,aw120_database_1,survey_camera_photo,AW120:4,3,100.0,100,0.0,0.0,0.0,86.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,100.0
3,low_76_0,2026,AW120,9343,DSCN9343_c.jpg,DSCN9343_c.jpg,South Photoplot,aw120_database_1,survey_camera_photo,AW120:5,4,100.0,100,0.0,0.0,0.0,86.0,0.0,12.0,0.0,0.0,2.0,0.0,0.0,100.0
4,low_76_0,2026,AW120,9344,DSCN9344_c.jpg,DSCN9344_c.jpg,West Photoplot,aw120_database_1,survey_camera_photo,AW120:6,5,100.0,100,0.0,0.0,0.0,85.0,0.0,4.0,0.0,0.0,11.0,0.0,0.0,100.0


,Plot,Year,Camera,n_photos,n_hits,ARTR_FG,BAREGROUND_FG,BIOCRUST_FG,EAG_FG,EF_FG,LITTER_FG,NPF_FG,OTHER_FG,PBG_FG,SHRUB_FG,SHRUB_total,total_cover_fg
0,20260527_mid_17_AGCR,2026,Cam5,5,500,0.0,3.4,0.0,61.6,0.4,13.2,0.0,0.0,21.4,0.0,0.0,100.0
1,20260527_mid_17_erna10,2026,Cam5,5,500,0.0,1.8,0.0,33.6,5.0,39.6,0.0,0.0,0.0,20.0,20.0,100.0
2,20260601_high_29_erna10,2026,Cam5,5,500,0.0,46.6,0.0,8.4,0.0,6.2,0.0,0.0,0.0,38.8,38.8,100.0
3,20260603_low76_fourwing1,2026,AW120,5,500,0.0,2.4,0.0,42.6,0.0,6.4,0.0,0.0,0.0,48.6,48.6,100.0
4,20260603_low76_fourwing2,2026,AW120,5,500,0.0,15.0,0.0,33.8,0.0,25.8,0.0,0.0,20.2,5.2,5.2,100.0


In [80]:
# =============================================================================
# BUILD SPECIES / SURFACE COVER
#
# Preserve point_resolved exactly as classified.
# Apply analytical species recodes only in the species-output layer.
# =============================================================================

point_species = point_resolved.copy()

SPECIES_RECODE_2026 = {
    "VUMI": "VULPI",
    "VUOC": "VULPI",
}

point_species["label"] = (
    point_species["label"]
    .replace(SPECIES_RECODE_2026)
)


# Canonical species/surface column order
species_columns = sorted(
    point_species["label"]
    .dropna()
    .unique()
)


# ---------------------------------------------------------------------
# Photo-level species/surface cover
# ---------------------------------------------------------------------

photo_species = build_photo_cover(
    point_species,
    class_col="label",
    class_order=species_columns,
)

photo_species["total_cover"] = (
    photo_species[species_columns]
    .sum(axis=1)
)


# ---------------------------------------------------------------------
# Site-level species/surface cover
# ---------------------------------------------------------------------

site_species = build_site_cover(
    point_species,
    class_col="label",
    class_order=species_columns,
)

site_species["total_cover"] = (
    site_species[species_columns]
    .sum(axis=1)
)


# ---------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------

if not np.allclose(
    photo_species["total_cover"],
    100.0,
    atol=1e-9,
):
    raise ValueError(
        "Photo species cover does not sum to 100."
    )

if not np.allclose(
    site_species["total_cover"],
    100.0,
    atol=1e-9,
):
    raise ValueError(
        "Site species cover does not sum to 100."
    )


vulpia_cols = [
    c
    for c in species_columns
    if c in ["VUMI", "VUOC", "VULPI"]
]

print("Vulpia species columns:", vulpia_cols)

if vulpia_cols != ["VULPI"]:
    raise ValueError(
        f"Vulpia canonicalization failed: {vulpia_cols}"
    )


print(
    f"Photo species: {photo_species.shape}"
)

print(
    f"Site species:  {site_species.shape}"
)

print(
    "PASS: VUMI/VUOC are represented as VULPI "
    "in species outputs."
)

Vulpia species columns: ['VULPI']
Photo species: (2269, 65)
Site species:  (454, 57)
PASS: VUMI/VUOC are represented as VULPI in species outputs.


## 7. Write primary outputs and identity audit

In [81]:
PHOTO_SPP_CSV = OUTPUT_DIR / "2026_SamplePoint_photo_species_cover.csv"
SITE_SPP_CSV = OUTPUT_DIR / "2026_SamplePoint_site_species_cover.csv"
PHOTO_FG_CSV = OUTPUT_DIR / "2026_SamplePoint_photo_functional_group_cover.csv"
SITE_FG_CSV = OUTPUT_DIR / "2026_SamplePoint_site_functional_group_cover.csv"
IDENTITY_AUDIT_CSV = OUTPUT_DIR / "2026_SamplePoint_identity_resolution_audit.csv"

photo_species.to_csv(PHOTO_SPP_CSV, index=False)
site_species.to_csv(SITE_SPP_CSV, index=False)
photo_fg.to_csv(PHOTO_FG_CSV, index=False)
site_fg.to_csv(SITE_FG_CSV, index=False)
resolution.to_csv(IDENTITY_AUDIT_CSV, index=False)

print("WROTE:")
for path in [
    PHOTO_SPP_CSV,
    SITE_SPP_CSV,
    PHOTO_FG_CSV,
    SITE_FG_CSV,
    IDENTITY_AUDIT_CSV,
]:
    print(" ", path)

WROTE:
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_photo_species_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_species_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_photo_functional_group_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_functional_group_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_identity_resolution_audit.csv


## 8. Build an append-ready 2026 site table

This section reads the historical `NCA_Master_Aligned_spp_fg.csv` only to establish its schema. It does **not** overwrite the historical master.

The append-ready table:

- preserves historical species-column order;
- fills historical species absent in 2026 with zero;
- adds genuinely new 2026 labels after the historical species columns;
- appends the same functional-group fields used historically;
- excludes the old `X` export-index column.

In [ ]:
APPEND_READY_CSV = OUTPUT_DIR / "2026_SamplePoint_site_spp_fg_append_ready.csv"

if NCA_MASTER.exists():
    legacy = pd.read_csv(NCA_MASTER, low_memory=False)

    required_legacy = {
        "Plot", "Year", "total_cover",
        *FG_COLUMNS, "SHRUB_total", "total_cover_fg",
    }
    missing_legacy = required_legacy - set(legacy.columns)
    if missing_legacy:
        raise ValueError(f"Legacy master missing: {sorted(missing_legacy)}")

    year_idx = legacy.columns.get_loc("Year")
    total_idx = legacy.columns.get_loc("total_cover")
    legacy_species_columns = list(legacy.columns[year_idx + 1 : total_idx])

    new_2026_species_columns = [
        c for c in species_columns
        if c not in legacy_species_columns
    ]

    print(f"Historical species/surface columns: {len(legacy_species_columns):,}")
    print(f"New 2026 species/surface columns:   {len(new_2026_species_columns):,}")
    print(new_2026_species_columns)

    site_2026 = (
        site_species.drop(columns=["Camera", "n_photos", "n_hits"])
        .merge(
            site_fg[
                [
                    "Plot", "Year", *FG_COLUMNS,
                    "SHRUB_total", "total_cover_fg",
                ]
            ],
            on=["Plot", "Year"],
            how="left",
            validate="one_to_one",
        )
    )

    union_species_columns = legacy_species_columns + new_2026_species_columns

    for col in union_species_columns:
        if col not in site_2026.columns:
            site_2026[col] = 0.0

    append_ready_2026 = site_2026[
        [
            "Plot", "Year",
            *union_species_columns,
            "total_cover",
            *FG_COLUMNS,
            "SHRUB_total",
            "total_cover_fg",
        ]
    ].copy()

    dup = append_ready_2026.duplicated(["Plot", "Year"], keep=False)
    if dup.any():
        display(append_ready_2026.loc[dup, ["Plot", "Year"]])
        raise ValueError("Duplicate 2026 Plot/Year rows in append-ready table.")

    legacy_keys = set(
        zip(legacy["Plot"].astype(str), legacy["Year"].astype(int))
    )
    collisions = append_ready_2026[
        append_ready_2026.apply(
            lambda r: (str(r["Plot"]), int(r["Year"])) in legacy_keys,
            axis=1,
        )
    ]

    print(
        f"2026 Plot/Year keys already present in historical master: "
        f"{len(collisions):,}"
    )
    if len(collisions):
        display(collisions[["Plot", "Year"]])

    append_ready_2026.to_csv(APPEND_READY_CSV, index=False)
    print("\nWrote:")
    print(APPEND_READY_CSV)
    display(append_ready_2026.head())

else:
    append_ready_2026 = None
    print(
        "Historical NCA master not found at configured paths. "
        "Core 2026 outputs are complete; update NCA_MASTER and rerun this cell later."
    )

Historical species/surface columns: 70
New 2026 species/surface columns:   16
['AMAC2', 'ARAR8', 'CADR', 'CICY', 'DISP', 'ERCI6', 'ERTR13', 'HAGL', 'PHHA', 'PHLO2', 'PIDE4', 'PSLA3', 'ROCK', 'SHIT', 'TRASH', 'VULPI']
2026 Plot/Year keys already present in historical master: 0

Wrote:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_spp_fg_append_ready.csv


,Plot,Year,ARTR2,CHVI8,ERNA10,KRLA2,ATCO,PUTR2,BAPR5,POSE,BRTE,AGCR,PSSP6,ACTH7,HECO,ELEL5,ACHY,ELYMUS,GRSP,GUSA,SAVE4,SHRUB,BRBR,PBG,ONAC,CHJU,FORB,BRASS,SATR12D,CETE5,LEPE2,NOTR,UNK.PLANT,OTHER,LITTER,BAREGROUND,BASC,SATR12,BIOCRUST,HEAN3,SIAL2D,LASE,SIAL2,CEST8,CHVI8D,DESO2,VUOC,ARTR2D,AMAL,TEGL,BAPR5D,LECI4,EPBR3,DEPI,TACA8,CREPI,FEID,EUMA7,ATCA2,SPGR2,PSJU3,BAAM4,TRDU,MACA2,GAYOP,SPCR,TRMA3,ATCOD,VUBR,TESP2,ATCA2D,BASA3,AMAC2,ARAR8,CADR,CICY,DISP,ERCI6,ERTR13,HAGL,PHHA,PHLO2,PIDE4,PSLA3,ROCK,SHIT,TRASH,VULPI,total_cover,ARTR_FG,BAREGROUND_FG,BIOCRUST_FG,EAG_FG,EF_FG,LITTER_FG,NPF_FG,OTHER_FG,PBG_FG,SHRUB_FG,SHRUB_total,total_cover_fg
0,20260527_mid_17_AGCR,2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.2,61.6,16.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.2,3.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,3.4,0.0,61.6,0.4,13.2,0.0,0.0,21.4,0.0,0.0,100.0
1,20260527_mid_17_erna10,2026,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,33.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,1.4,0.0,0.0,0.0,0.0,0.0,38.2,1.8,0.0,0.0,0.0,0.0,0.0,0.0,4.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,1.8,0.0,33.6,5.0,39.6,0.0,0.0,0.0,20.0,20.0,100.0
2,20260601_high_29_erna10,2026,0.0,0.0,38.8,0.0,0.0,0.0,0.0,0.0,8.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,46.6,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,46.6,0.0,8.4,0.0,6.2,0.0,0.0,0.0,38.8,38.8,100.0
3,20260603_low76_fourwing1,2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.4,2.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,2.4,0.0,42.6,0.0,6.4,0.0,0.0,0.0,48.6,48.6,100.0
4,20260603_low76_fourwing2,2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.2,33.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.8,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,15.0,0.0,33.8,0.0,25.8,0.0,0.0,20.2,5.2,5.2,100.0


: 

## 9. Final QA

In [77]:
qa = {
    "source_classification_rows": int(len(samplepoint_photos)),
    "included_analytical_photos": int(len(included_resolution)),
    "excluded_stale_source_rows": int(len(excluded_resolution)),
    "analytical_point_hits": int(len(point_resolved)),
    "unique_sites": int(point_resolved["Plot"].nunique()),
    "photo_species_total_min": float(photo_species["total_cover"].min()),
    "photo_species_total_max": float(photo_species["total_cover"].max()),
    "site_species_total_min": float(site_species["total_cover"].min()),
    "site_species_total_max": float(site_species["total_cover"].max()),
    "photo_fg_total_min": float(photo_fg["total_cover_fg"].min()),
    "photo_fg_total_max": float(photo_fg["total_cover_fg"].max()),
    "site_fg_total_min": float(site_fg["total_cover_fg"].min()),
    "site_fg_total_max": float(site_fg["total_cover_fg"].max()),
    "minimum_photos_per_site": int(site_species["n_photos"].min()),
    "maximum_photos_per_site": int(site_species["n_photos"].max()),
    "low_43_33_present": bool(
        site_species["Plot"].astype(str).eq("low_43_33").any()
    ),
    "aw120_9915_present": bool(
        (
            included_resolution["Camera"].eq("AW120")
            & included_resolution["PhotoNumber"].eq(9915)
        ).any()
    ),
}

print("=" * 78)
print("FINAL 2026 SAMPLEPOINT COVER QA")
print("=" * 78)

for key, value in qa.items():
    print(f"{key:34s}: {value}")

print("\nPhoto-count distribution:")
display(
    site_species["n_photos"]
    .value_counts()
    .sort_index()
    .rename_axis("n_photos")
    .reset_index(name="n_sites")
)

print("\nPrimary outputs:")
for path in [
    PHOTO_SPP_CSV,
    SITE_SPP_CSV,
    PHOTO_FG_CSV,
    SITE_FG_CSV,
    IDENTITY_AUDIT_CSV,
]:
    print(" ", path)

if NCA_MASTER.exists():
    print(" ", APPEND_READY_CSV)

FINAL 2026 SAMPLEPOINT COVER QA
source_classification_rows        : 2269
included_analytical_photos        : 2269
excluded_stale_source_rows        : 0
analytical_point_hits             : 226900
unique_sites                      : 454
photo_species_total_min           : 99.99999999999999
photo_species_total_max           : 100.00000000000001
site_species_total_min            : 99.99999999999997
site_species_total_max            : 100.00000000000003
photo_fg_total_min                : 100.0
photo_fg_total_max                : 100.00000000000001
site_fg_total_min                 : 99.99999999999997
site_fg_total_max                 : 100.00000000000003
minimum_photos_per_site           : 4
maximum_photos_per_site           : 6
low_43_33_present                 : True
aw120_9915_present                : True

Photo-count distribution:


,n_photos,n_sites
0,4,2
1,5,451
2,6,1



Primary outputs:
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_photo_species_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_species_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_photo_functional_group_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_functional_group_cover.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_identity_resolution_audit.csv
  D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Aggregated\2026_SamplePoint_site_spp_fg_append_ready.csv


In [78]:
# Show the three non-standard sites
display(
    site_species.loc[
        site_species["n_photos"] != 5,
        ["Plot", "Camera", "n_photos", "n_hits"]
    ]
    .sort_values(["n_photos", "Plot"])
)

,Plot,Camera,n_photos,n_hits
31,A3_23,AW120,4,400
369,mid_16_33,AW120,4,400
27,A3_12,AW120,6,600


In [79]:
vulpia_cols = [
    c for c in photo_species.columns
    if c in ["VUMI", "VUOC", "VULPI"]
]

print("Vulpia species columns:", vulpia_cols)

Vulpia species columns: ['VUMI', 'VUOC']
